## GraphRAG with a LoRA llm adapter applied to healthcare benefits
### Background
Questions involving contracts or benefits are particularly difficult to answer through a traditional RAG system which does the standard chunking, embedding, retrieval chain. What often happens is at retrieval, the agent fails to collect all relevant documents related to the query. For example, the agent verifies the diagnosis, links it to a procedure for treatment, sees that the procedure is covered by a policy, but fails to collect the exception clause which says they do not cover the procedure for that specific diagnosis. <br><br>
GraphRAG seeks to solve this by defining the key nodes (policies, diagnoses, procedures, exceptions), builds a graph database of the relationships between them, and at retrieval time ensuring each element of the chain supports the claim.<br><br>
Further, since healthcare is a highly regulated environment, we also need the response to be clear and consistently formatted.
### Approach
This demo builds a GraphRAG of healthcare benefits data and verifies through a series of benefits-related questions. The GraphRAG output includes all of the traceability info related to the question. To generate a consistent response, we train a LoRA to a 7B weight llm with a series of questions, the graph output, and the ideal final response.

### Execution Mode, Imports, Globals & Paths

In [ ]:
# MODE can be: "RAG_ONLY", "LORA_TRAIN", "LORA_ONLY", "COMPARE"
MODE = "COMPARE"

# Paths
from pathlib import Path
ROOT = Path("/Users/douglasdaly/Documents/GitHub/Generative-AI/notebooks")
HOME = ROOT.parent / "assets" / "graph_rag_demo"

# Imports
import sys
import torch
import yaml
import pandas as pd
from datasets import Dataset
sys.path.append(str(HOME))
from graph.engine import GraphRAG
from graph.adapters_sets import add_sets_and_policies

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, LoraConfig, get_peft_model


# Globals
BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
ADAPTER_DIR = "lora_coverage_adapter"
device = "mps" if torch.backends.mps.is_available() else "cpu"
IGNORE_INDEX = -100
MAX_LEN = 1024





### Show file tree

In [2]:
from support import tree_markdown
display(tree_markdown(HOME))

```graph_rag_demo/
├── data
│   ├── cms_coverage
│   │   ├── arthrocentesis.md
│   │   └── nerve_conduction.md
│   ├── Code-desciptions-April-2025
│   │   ├── icd10-Order-Files-April-2025.pdf
│   │   ├── icd10cm-codes-addenda-April-2025.txt
│   │   ├── icd10cm-codes-April-2025.txt
│   │   ├── icd10cm-Codes-File-April-2025.pdf
│   │   ├── icd10cm-order-addenda-April-2025.txt
│   │   └── icd10cm-order-April-2025.txt
│   ├── docs
│   │   ├── clinic_policy.txt
│   │   ├── cms_arthrocentesis.txt
│   │   └── cms_ncs.txt
│   ├── clinic_policy.md
│   ├── coverage.yaml
│   ├── edges_policies.yaml
│   ├── edges_taxonomy.csv
│   ├── icd10.csv
│   ├── modifiers.csv
│   ├── policy_sets.yaml
│   ├── procedures.csv
│   ├── rules.yaml
│   └── synonyms.csv
├── graph
│   ├── __init__.py
│   ├── adapters_sets.py
│   ├── build.py
│   ├── engine.py
│   ├── ground.py
│   ├── ingest.py
│   ├── params.py
│   ├── reason.py
│   ├── registry.py
│   ├── render.py
│   └── rules.py
├── tests
├── __init__.py
├── lora_gold_coverage.csv
├── node_docs.yaml
└── notes.txt
```

### Load GraphRAG

In [3]:
# Define the Graph RAG object -- all relevant data and code are in assets/graph_rag_deo
eng = GraphRAG(HOME / "node_docs.yaml")

# Load policies
with open(HOME / "data"/ "policy_sets.yaml", "r") as f:
    policy_sets = yaml.safe_load(f)
add_sets_and_policies(eng, policy_sets)


def answer_with_graph(q: str) -> str:
    return eng.answer(q)

### Demo - show GraphRAG results from several test cases

In [4]:
if MODE in ["RAG_ONLY", "COMPARE"]:
    test_questions = [
        "Is arthrocentesis (20610) covered for M75.11?",
        "Is 95900 covered for E11.40?",
        "Is arthrocentesis covered for frozen shoulder?",
    ]
    for q in test_questions:
        print("Q:", q)
        print(answer_with_graph(q))
        print("-" * 80)


Q: Is arthrocentesis (20610) covered for M75.11?
A: Yes — covered when medically necessary for the indicated diagnosis.

Per-diagnosis:
  M75.11: ✓ qualifies under cms-arthro via DxSet[icd:rotator_cuff]
  M7511: ✓ qualifies under cms-arthro via DxSet[icd:rotator_cuff]

Why (reason path):
  Diagnosis:M7511 --IS_IN→ DxSet:icd:rotator_cuff
  Diagnosis:M75.11 --IS_IN→ DxSet:icd:rotator_cuff
  Procedure:20610 --IS_IN→ ProcSet:cpt:arthro_block
  Coverage:cms-arthro --USES_PROCSET→ ProcSet:cpt:arthro_block
  Coverage:cms-arthro --USES_DXSET→ DxSet:icd:rotator_cuff
  Coverage:cms-tendon --USES_DXSET→ DxSet:icd:rotator_cuff

Matched sets:
  cms-arthro: ProcSet[cpt:arthro_block] & DxSet[icd:rotator_cuff]

Citations:
- cms-arthro: CMS Coverage — Arthrocentesis
- cms-tendon: CMS Coverage — Tendon Injection
--------------------------------------------------------------------------------
Q: Is 95900 covered for E11.40?
A: Yes — covered when medically necessary for the indicated diagnosis.

Why (reas

### Load Gold dataset and define system prompt for improved answerer

In [33]:
if MODE in ["LORA_TRAIN", "LORA_ONLY", "COMPARE"]:
    import json
    PROMPT_TEMPLATE = """You are a coverage decision assistant. You do NOT invent policies.
        You ONLY make decisions based on the graph evidence provided.

        Return an answer that has:
        1. A clear decision: COVERED, NOT_COVERED, or NEEDS_HUMAN_REVIEW.
        2. A short reason grounded in the configured procedure sets, diagnosis sets, and policies.
        3. A standard disclaimer at the end.

        Question: {question}

        GraphRAG analysis:
        {graph_output}

        Respond in this JSON format:
        {{
        "decision": "...",
        "reason": "...",
        "disclaimer": "..."
        }}
        
        Now produce the answer."""
    
    def build_target(row):
        '''Gold response based on data listing decision, reason and disclaimer'''
        obj = {
            "decision": row["decision"],
            "reason": row["reason"],
            "disclaimer": row["disclaimer"],
        }
        # compact JSON so the model learns a consistent format
        return json.dumps(obj, ensure_ascii=False)

    def build_prompt(question: str | None = None,
                graph_output: str | None = None,
                row = None) -> str:
        '''Define prompt for LLM to create response'''
        if row is not None:
          question = row['question']
          graph_output = row['graph_output']
        return PROMPT_TEMPLATE.format(
            question=question,
            graph_output=graph_output,
        )
    gold_df = pd.read_csv(HOME / "lora_gold_coverage.csv")
    gold_df["target"] = gold_df.apply(build_target, axis=1)
    gold_df["prompt"] = gold_df.apply(lambda row:build_prompt(row=row), axis=1)

    gold_ds = Dataset.from_pandas(gold_df[["prompt", "target"]])


### Define Tokenizer & tokenized dataset for LoRA

In [34]:
if MODE in ["LORA_TRAIN"]:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    def tokenize_example(example):
        prompt = example["prompt"]
        target = example["target"]

        
        full_text = prompt + "\n\n" + target

        tok = tokenizer(
            full_text,
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
        )
        input_ids = tok["input_ids"]
        attention_mask = tok["attention_mask"]

        prompt_ids = tokenizer(prompt + "\n\n", add_special_tokens=False)["input_ids"]
        prompt_len = min(len(prompt_ids), MAX_LEN)

        labels = input_ids.copy()
        labels[:prompt_len] = [IGNORE_INDEX] * prompt_len

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

    tokenized_ds = gold_ds.map(
        tokenize_example,
        batched=False,
        remove_columns=gold_ds.column_names,
    )


### Load base LLM + LoRA for training

In [35]:
if MODE == "LORA_TRAIN":
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16 if device == "mps" else torch.float32,
        low_cpu_mem_usage=True,
    ).to(device)

    lora_cfg = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(base_model, lora_cfg)
    model.print_trainable_parameters()


### Train LoRA

In [36]:
if MODE == "LORA_TRAIN":
    from transformers import TrainingArguments, Trainer

    training_args = TrainingArguments(
        output_dir=ADAPTER_DIR,
        num_train_epochs=5,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        warmup_ratio=0.1,
        logging_steps=5,
        save_strategy="epoch",
        fp16=False,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds,
    )
    trainer.train()

    model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)

    # Memory clean-up after training
    # Drops the last Python references (del).
    # Forces garbage collection (gc.collect()).
    # Asks PyTorch to release cached MPS memory (empty_cache()
    del trainer
    del model
    import gc, torch
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()


### Load LoRA model for inference

In [41]:
if MODE in ["LORA_ONLY", "COMPARE"]:
    tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16 if device == "mps" else torch.float32,
        low_cpu_mem_usage=True,
    ).to(device)

    lora_model = PeftModel.from_pretrained(
        base_model,
        ADAPTER_DIR,
        is_trainable=False,
    ).to(device)
    lora_model.eval()

    def generate_coverage_answer(question: str):
        # 1) get graph answer
        graph_output = eng.answer(question)
        prompt = build_prompt(question=question, graph_output=graph_output)

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_LEN,
        ).to(device)

        with torch.no_grad():
            out = lora_model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
            )

        text = tokenizer.decode(out[0], skip_special_tokens=True)
        return graph_output, text


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

### Compare RAG vs LoRA

In [42]:
if MODE == "COMPARE":
    compare_questions = [
        "Is 95900 covered for E11.40?",
        "Is arthrocentesis covered for frozen shoulder?",
        "Is 95900 covered for R20.0?",
        "Is arthrocentesis (20610) covered for M75.01 and M75.41?",
    ]

    for q in compare_questions:
        print("Q:", q)
        graph_only = eng.answer(q)
        graph_ans, lora_out = generate_coverage_answer(q)

        print("\n--- GraphRAG answer ---")
        print(graph_only)
        print("\n--- GraphRAG + LoRA assistant ---")
        print(lora_out)
        print("=" * 100)


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Q: Is 95900 covered for E11.40?


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



--- GraphRAG answer ---
A: Yes — covered when medically necessary for the indicated diagnosis.

Why (reason path):
  Diagnosis:E1140 --IS_IN→ DxSet:icd:diabetic_neuropathy
  Procedure:95900 --IS_IN→ ProcSet:cpt:ncs
  Coverage:cms-ncs --USES_PROCSET→ ProcSet:cpt:ncs
  Coverage:cms-ncs --USES_DXSET→ DxSet:icd:diabetic_neuropathy

Matched sets:
  cms-ncs: ProcSet[cpt:ncs] & DxSet[icd:diabetic_neuropathy]

Citations:
- cms-ncs: CMS Coverage — Nerve Conduction Studies
- icd10: E1140 — Type 2 diabetes mellitus with diabetic neuropathy, unspecified

--- GraphRAG + LoRA assistant ---
You are a coverage decision assistant. You do NOT invent policies.
        You ONLY make decisions based on the graph evidence provided.

        Return an answer that has:
        1. A clear decision: COVERED, NOT_COVERED, or NEEDS_HUMAN_REVIEW.
        2. A short reason grounded in the configured procedure sets, diagnosis sets, and policies.
        3. A standard disclaimer at the end.

        Question: Is 959

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



--- GraphRAG answer ---
A: Yes — covered when medically necessary for the indicated diagnosis.

Per-diagnosis:
  M750*: ✓ qualifies under cms-arthro via DxSet[icd:frozen_shoulder]
  M7500: ✗ no matching policy set
  M7501: ✗ no matching policy set
  M7502: ✗ no matching policy set

Why (reason path):
  Diagnosis:M750* --IS_IN→ DxSet:icd:frozen_shoulder
  Procedure:20610 --IS_IN→ ProcSet:cpt:arthro_block
  Coverage:cms-arthro --USES_PROCSET→ ProcSet:cpt:arthro_block
  Coverage:cms-arthro --USES_DXSET→ DxSet:icd:frozen_shoulder
  Diagnosis:M7500 --IS_A→ Diagnosis:M750*
  Diagnosis:M7501 --IS_A→ Diagnosis:M750*
  Diagnosis:M7502 --IS_A→ Diagnosis:M750*

Matched sets:
  cms-arthro: ProcSet[cpt:arthro_block] & DxSet[icd:frozen_shoulder]

Citations:
- cms-arthro: CMS Coverage — Arthrocentesis
- icd10: M7500 — Adhesive capsulitis of unspecified shoulder
- icd10: M7501 — Adhesive capsulitis of right shoulder
- icd10: M7502 — Adhesive capsulitis of left shoulder

--- GraphRAG + LoRA assistant 

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



--- GraphRAG answer ---
A: Coverage not found for this pairing in the local graph.

Why (reason path):
  Procedure:95900 --IS_IN→ ProcSet:cpt:ncs
  Coverage:cms-ncs --USES_PROCSET→ ProcSet:cpt:ncs

Matched sets:

Citations:
- cms-ncs: CMS Coverage — Nerve Conduction Studies
- icd10: R200 — Anesthesia of skin

--- GraphRAG + LoRA assistant ---
You are a coverage decision assistant. You do NOT invent policies.
        You ONLY make decisions based on the graph evidence provided.

        Return an answer that has:
        1. A clear decision: COVERED, NOT_COVERED, or NEEDS_HUMAN_REVIEW.
        2. A short reason grounded in the configured procedure sets, diagnosis sets, and policies.
        3. A standard disclaimer at the end.

        Question: Is 95900 covered for R20.0?

        GraphRAG analysis:
        A: Coverage not found for this pairing in the local graph.

Why (reason path):
  Procedure:95900 --IS_IN→ ProcSet:cpt:ncs
  Coverage:cms-ncs --USES_PROCSET→ ProcSet:cpt:ncs

Matche

## Archive - no need to run

#### Download ICD code descriptions

In [ ]:
if 0:
    import re
    from pathlib import Path
    import pandas as pd
    import requests

    # 1) Download the CMS ZIP (adjust URL to the current year's "Code Descriptions in Tabular Order")
    url = "https://ftp.cdc.gov/pub/health_statistics/nchs/publications/ICD10CM/2025-Update/Code-desciptions-April-2025.zip"
    zbytes = requests.get(url, timeout=60).content
    # Then unzip


In [ ]:
if 0:
    IN_PATH  = HOME / "data" / "Code-desciptions-April-2025" / "icd10cm-codes-April-2025.txt"
    OUT_PATH = HOME / "data" / "icd10.csv"

    # Your exact requirement for the code: [A-Z][0-9]+
    # Separator is 2+ spaces, second column is the remainder.
    LINE_RE = re.compile(r'^\s*([^"]*?)\s*$')  # first pass: strip quotes safely
    SPLIT_RE = re.compile(r'^\s*([A-Z0-9\.]+)\s{1,}(.*?)\s*$')

    def parse_icd_lines(in_path: Path):
        rows = []
        skipped = 0
        with in_path.open("r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                # Remove all double quotes and surrounding whitespace/newlines
                line = line.replace('"', '')
                if 1:
                    m = SPLIT_RE.match(line)
                    if not m:
                        print(line)
                        skipped += 1
                        continue
                    code, desc = m.group(1), m.group(2)
                    rows.append((code, desc))
                else:
                    rows.append(line)
        return rows, skipped

    def main():
        rows, skipped = parse_icd_lines(IN_PATH)
        if not rows:
            raise SystemExit(f"No rows parsed from {IN_PATH}. Check the input format.")
        df = pd.DataFrame(rows, columns=["code", "name"])
        OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(OUT_PATH, sep="|", index=False, encoding="utf-8")
        print(f"Wrote {len(df):,} rows to {OUT_PATH}")
        if skipped:
            print(f"Skipped {skipped:,} lines that didn’t match the pattern (ok if the file has headers/footers).")

    if __name__ == "__main__":
        main()


### Define tests for "Gold test set"

In [ ]:
if 0:
    tests = [
    "Is arthrocentesis (20610) covered for M75.01 and M75.41?",
    "Is joint injection (20552) covered for M75.41?",
    "Bilateral arthrocentesis (20610) for M75.01 — any modifier?",
    "Is arthrocentesis (20610) covered for M54.2?",
    "Is 20610 covered for M75.11?",
    "Is 20552 covered for M75.11?",
    "Is nerve conduction study (95900) covered for R20.0 (numbness) under clinic policy?",
    "Is 95900 covered for E11.40?",
    "Is 95900 covered for M75.01?",
    "Is arthrocentesis covered for frozen shoulder?",
    "Is joint injection covered for a rotator cuff tear?",
    "Is nerve conduction study covered for neck pain?",
    "Is arthrocentesis covered for a rotator cuff tear?",
    "Is joint injection covered for shoulder impingement?",
    "Is 20552 covered for neck pain?",
    "Is 20610 covered for cervicalgia?",
    "Is 95900 covered for diabetic neuropathy?",
    "Is joint tap covered for M54.2?",
    "Is arthrocentesis covered for M75.01?"
    "Is arthrocentesis covered for frozen shoulder and rotator cuff tear?",
    "Is joint injection covered for M54.2 and M75.01?",
    "Is 95900 covered for M75.01 and M75.41?",
    "Is arthrocentesis covered for M75.11 and R20.0?"
    ]

    for q in tests:
        print("Q:", q)
        print(eng.answer(q), "\n"+"-"*80+"\n")

### Build HF dataset from CSV

In [ ]:
if 0:
    import pandas as pd
    from datasets import Dataset

    PROMPT_TEMPLATE = """You are a coverage decision assistant. You do NOT invent policies.
    You ONLY make decisions based on the graph evidence provided.

    Return an answer that has:
    1. A clear decision: COVERED, NOT_COVERED, or NEEDS_HUMAN_REVIEW.
    2. A short reason grounded in the configured procedure sets, diagnosis sets, and policies.
    3. A standard disclaimer at the end.

    Question:
    {question}

    Graph evidence:
    {graph_output}

    Now produce the answer."""

    def build_prompt(row):
        return PROMPT_TEMPLATE.format(
            question=row["question"],
            graph_output=row["graph_output"],
        )

    df["prompt"] = df.apply(build_prompt, axis=1)

    dataset = Dataset.from_pandas(df[["prompt", "target"]])
    train_dataset = dataset.shuffle(seed=42)
